# Tutorial 3 -- Pipelines

A **pipeline** is a single-argument callable `Pipeline = Callable[[DataSource], DataSource]`. Recipes are pipelines built once with concrete params; you can also build your own from ops, or load one from a YAML template.

This tutorial covers three forms, in order of explicitness:

1. **Python `compose`**: assemble a pipeline at runtime from ops.
2. **YAML template**: configure the same pipeline in a text file.
3. **Custom op registration**: extend the pipeline vocabulary.

## 1. Compose a pipeline in Python

`compose(*ops)` returns a `Pipeline`. Each `op` is a single-arg callable -- bind op-specific params via `functools.partial`.

In [1]:
from functools import partial
import numpy as np

from cfdmod import (
    SurfaceDataSource,
    TimeAxis,
    Topology,
    ElementMeta,
    MemoryFieldStore,
    compose,
)
from cfdmod.ops.field import moving_average, MovingAverageParams, scale, ScaleParams
from cfdmod.ops.data_source_create import compute_statistics, StatisticsParams

rng = np.random.default_rng(0)
verts = np.array([[0, 0, 0], [1, 0, 0], [0, 1, 0]], dtype=np.float64)
tris = np.array([[0, 1, 2]], dtype=np.int32)
p = rng.normal(100.0, 8.0, size=(1, 1000))

raw = SurfaceDataSource(
    time=TimeAxis(initial_time=0.0, timestep_size=0.001, n_timesteps=1000),
    topology=Topology.triangles(tris, verts),
    elements=ElementMeta(),
    fields=MemoryFieldStore({"pressure": p}),
)

pipe = compose(
    partial(moving_average, p=MovingAverageParams(field="pressure", window=0.05, out="p_smooth")),
    partial(scale, p=ScaleParams(field="p_smooth", factor=0.01)),
    partial(
        compute_statistics, p=StatisticsParams(field="p_smooth", kinds=["mean", "rms", "peak_max"])
    ),
)
out = pipe(raw)
print("fields after pipeline:", out.field_names)
print("time aggregated:      ", out.time.is_time_aggregated)
for f in out.field_names:
    print(f"  {f:>10} = {out.fields.read(f)[0]:.4f}")

fields after pipeline: ['mean', 'rms', 'peak_max']
time aggregated:       True
        mean = 0.9962
         rms = 0.0105
    peak_max = 1.0233


## 2. Same pipeline, declared in YAML

Templates are flat lists of *steps*. Each step has a `kind` (the op), a `source` (a previous step's `id` or an `inputs:` key), plus op-specific fields. Outputs are written via the supplied `Storage`.

In [2]:
import pathlib, tempfile, textwrap

yaml_template = textwrap.dedent("""
    name: smoothed_stats
    inputs:
      raw:
        kind: surface
        path: raw
    pipeline:
      - id: p_smooth
        kind: moving_average
        source: raw
        field: pressure
        window: 0.05
        out: p_smooth
      - id: p_scaled
        kind: scale
        source: p_smooth
        field: p_smooth
        factor: 0.01
      - id: stats
        kind: statistics
        source: p_scaled
        field: p_smooth
        kinds: [mean, rms, peak_max]
    outputs:
      result:
        source: stats
        path: stats_out
    """)

with tempfile.TemporaryDirectory() as tmp:
    tmp_path = pathlib.Path(tmp)
    yaml_path = tmp_path / "pipe.yaml"
    yaml_path.write_text(yaml_template)

    from cfdmod import MemoryStorage, load_template, run_template

    storage = MemoryStorage()
    storage.write_data_source(str(tmp_path / "raw"), raw)

    template = load_template(yaml_path)
    bindings = run_template(template, storage=storage)

    stats = bindings["stats"]
    print("fields:", stats.field_names)
    for f in stats.field_names:
        print(f"  {f:>10} = {stats.fields.read(f)[0]:.4f}")

fields: ['mean', 'rms', 'peak_max']
        mean = 0.9962
         rms = 0.0105
    peak_max = 1.0233


Numerically identical to the Python pipeline above -- the YAML loader dispatches each step into the same op functions.

## 3. Register a custom op

Need an op the registry doesn't ship? Define a pure function, declare a Pydantic params model, and `register_op` it. Users (and YAML templates) can then reference your `kind`.

In [3]:
from typing import ClassVar, Literal
from cfdmod import register_op, PipelineTemplate, run_template
from cfdmod.ops import OpParams
from cfdmod.core.data_source import DataSource


class ClipParams(OpParams):
    """Clamp a field's values into [low, high]."""

    kind: Literal["clip"] = "clip"
    field: str
    low: float
    high: float
    chunkable_along: ClassVar[frozenset[str]] = frozenset({"time", "elements"})


def clip(ds: DataSource, p: ClipParams) -> DataSource:
    arr = ds.fields.read(p.field)
    return ds.with_field(p.field, np.clip(arr, p.low, p.high))


register_op("clip", clip, ClipParams, arity="unary")

# Use it from YAML.
tpl = PipelineTemplate(
    inputs={"raw": {"kind": "surface", "path": "raw"}},
    pipeline=[
        {
            "id": "clipped",
            "kind": "clip",
            "source": "raw",
            "field": "pressure",
            "low": 90.0,
            "high": 110.0,
        }
    ],
)
storage = MemoryStorage()
storage.write_data_source("raw", raw)
bindings = run_template(tpl, storage=storage)
clipped = bindings["clipped"].fields.read("pressure")
print(f"clipped range: [{clipped.min():.2f}, {clipped.max():.2f}]")

clipped range: [90.00, 110.00]


## 4. Why YAML?

- **Same pipeline runs in tests (MemoryStorage) and in production (XdmfH5Storage)** -- no code change.
- **Templates are diffable**: changes to a Cp / Cf / S1 recipe show up in git.
- **Composable**: drop in custom ops without forking the recipe.
- **Reproducible**: the template *is* the recipe definition.

See also: `cfdmod.recipes.cp.build_cp` -- a Python builder that produces an equivalent pipeline programmatically.

Next: **Tutorial 4 -- Containers** (running the same pipeline over many wind directions / cases).